In [1]:
import pandas as pd

# Update the path to your actual raw CAN CSV file
file_path = '/Users/edu/Edu/testproject/tenderpilot_data/data/export_CAN_2023_2018.csv'

print("Loading raw CAN dataset...")
df_raw = pd.read_csv(file_path, low_memory=False)

print(f"Raw dataset shape: {df_raw.shape}")

print("\nRaw INFO_ON_NON_AWARD distribution:")
# dropna=False is critical because NaN means 'Awarded' in TED data
print(df_raw['INFO_ON_NON_AWARD'].value_counts(dropna=False))

Loading raw CAN dataset...
Raw dataset shape: (6198063, 75)

Raw INFO_ON_NON_AWARD distribution:
INFO_ON_NON_AWARD
NaN                         5212924
PROCUREMENT_UNSUCCESSFUL     794911
PROCUREMENT_DISCONTINUED     190228
Name: count, dtype: int64


In [2]:
import pandas as pd

print("1. Deduplicating by ID_AWARD to preserve true market structure...")
df_unique = df_raw.drop_duplicates(subset=['ID_AWARD']).copy()

print("2. Creating the Target...")
df_unique['TARGET_NOT_AWARDED'] = df_unique['INFO_ON_NON_AWARD'].notna().astype(int)

print("3. Balancing the dataset (250k Awarded / 250k Failed)...")
df_awarded = df_unique[df_unique['TARGET_NOT_AWARDED'] == 0]
df_failed = df_unique[df_unique['TARGET_NOT_AWARDED'] == 1]

df_awarded_sample = df_awarded.sample(n=250000, random_state=42)
df_failed_sample = df_failed.sample(n=250000, random_state=42)
df_balanced = pd.concat([df_awarded_sample, df_failed_sample]).sample(frac=1, random_state=42).reset_index(drop=True)

print("4. Selecting your 13 safe features + LOTS_NUMBER...")
columns_to_load = [
    # Required Identifiers
    'ID_AWARD', 'INFO_ON_NON_AWARD',
    
    # 13 Safe Pre-Award Features
    'B_MULTIPLE_CAE', 'B_EU_FUNDS', 'TOP_TYPE', 'ISO_COUNTRY_CODE', 
    'B_FRA_AGREEMENT', 'B_GPA', 'YEAR', 'TYPE_OF_CONTRACT', 
    'CAE_TYPE', 'CRIT_CODE', 'B_ACCELERATED', 'MAIN_ACTIVITY', 
    'CRIT_PRICE_WEIGHT',
    
    # The Safe Structural Driver
    'LOTS_NUMBER',
    
    # The engineered target
    'TARGET_NOT_AWARDED'
]

existing_cols = [c for c in columns_to_load if c in df_balanced.columns]
df_ultimate = df_balanced[existing_cols].copy()

print("5. Categorical Clean Up...")
if 'ISO_COUNTRY_CODE' in df_ultimate.columns:
    df_ultimate['ISO_COUNTRY_CODE'] = df_ultimate['ISO_COUNTRY_CODE'].fillna('UNKNOWN')

print("6. Dropping Identifiers to finalize matrix...")
cols_to_drop = ['ID_AWARD', 'INFO_ON_NON_AWARD']
existing_drops = [c for c in cols_to_drop if c in df_ultimate.columns]
df_ultimate = df_ultimate.drop(columns=existing_drops)

num_rows, num_cols = df_ultimate.shape
print(f"\nFinal Dataset Shape: {num_rows} rows, {num_cols} features.")
print(f"Final Features: {list(df_ultimate.columns)}")

output_file = 'f_balanced_500k.csv'
df_ultimate.to_csv(output_file, index=False)
print(f"Success! Data saved to {output_file}")

1. Deduplicating by ID_AWARD to preserve true market structure...
2. Creating the Target...
3. Balancing the dataset (250k Awarded / 250k Failed)...
4. Selecting your 13 safe features + LOTS_NUMBER...
5. Categorical Clean Up...
6. Dropping Identifiers to finalize matrix...

Final Dataset Shape: 500000 rows, 15 features.
Final Features: ['B_MULTIPLE_CAE', 'B_EU_FUNDS', 'TOP_TYPE', 'ISO_COUNTRY_CODE', 'B_FRA_AGREEMENT', 'B_GPA', 'YEAR', 'TYPE_OF_CONTRACT', 'CAE_TYPE', 'CRIT_CODE', 'B_ACCELERATED', 'MAIN_ACTIVITY', 'CRIT_PRICE_WEIGHT', 'LOTS_NUMBER', 'TARGET_NOT_AWARDED']
Success! Data saved to f_balanced_500k.csv


In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

print("1. Loading the Ultimate Dataset for Evaluation...")
df_eval = pd.read_csv('f_balanced_500k.csv', low_memory=False)
num_rows, num_cols = df_eval.shape

print("\n" + "="*50)
print("📊 STEP 1: MISSINGNESS & HALLUCINATION AUDIT")
print("="*50)
# Calculate the percentage of missing values per column
missing_stats = (df_eval.isnull().sum() / num_rows) * 100
missing_stats = missing_stats[missing_stats > 0].sort_values(ascending=False)

if missing_stats.empty:
    print("Zero missing values detected. (Warning: Could indicate artificial imputation!)")
else:
    print("Natural missing values detected (Proof of no EU '_FIN_' hallucinations):")
    for col, pct in missing_stats.items():
        print(f" - {col}: {pct:.2f}% missing")

print("\n" + "="*50)
print("📊 STEP 2: CATEGORICAL CARDINALITY PROFILING")
print("="*50)
# Separate categoricals to check if they will explode our One-Hot Encoder
cat_cols = df_eval.select_dtypes(include=['object', 'bool']).columns
print("Unique values per categorical feature:")
for col in cat_cols:
    unique_count = df_eval[col].nunique(dropna=False)
    print(f" - {col}: {unique_count} unique values")

print("\n" + "="*50)
print("📊 STEP 3: CORRELATION & TARGET LEAKAGE CHECK")
print("="*50)
# Convert Y/N flags to 1/0 for correlation testing
df_corr = df_eval.copy()
binary_mapping = {'Y': 1, 'N': 0}
for col in ['B_MULTIPLE_CAE', 'B_EU_FUNDS', 'B_FRA_AGREEMENT', 'B_GPA', 'B_ACCELERATED']:
    if col in df_corr.columns:
        df_corr[col] = df_corr[col].map(binary_mapping)

# Calculate correlations against the target
numeric_cols = df_corr.select_dtypes(include=[np.number]).columns
correlations = df_corr[numeric_cols].corr()['TARGET_NOT_AWARDED'].drop('TARGET_NOT_AWARDED').sort_values(ascending=False)

print("Correlation with TARGET_NOT_AWARDED (Looking for leaks > 0.80):")
leak_detected = False
for col, corr_val in correlations.items():
    print(f" - {col}: {corr_val:.4f}")
    if abs(corr_val) > 0.80:
        leak_detected = True

if leak_detected:
    print("\n🚨 WARNING: High correlation detected! Possible data leak remains.")
else:
    print("\n✅ SUCCESS: No suspicious correlations detected. Target is safe from post-award leaks.")

print("\n" + "="*50)
print("📊 STEP 4: FRAMEWORK AGREEMENT DISTORTION")
print("="*50)
if 'B_FRA_AGREEMENT' in df_eval.columns:
    fra_dist = df_eval['B_FRA_AGREEMENT'].value_counts(normalize=True, dropna=False) * 100
    print("Dataset distribution of Framework Agreements:")
    print(fra_dist.to_string())
else:
    print("Framework Agreement feature not found.")
print("="*50)

1. Loading the Ultimate Dataset for Evaluation...

📊 STEP 1: MISSINGNESS & HALLUCINATION AUDIT
Natural missing values detected (Proof of no EU '_FIN_' hallucinations):
 - B_ACCELERATED: 97.29% missing
 - CRIT_PRICE_WEIGHT: 70.91% missing
 - CRIT_CODE: 6.79% missing
 - B_GPA: 5.56% missing
 - B_EU_FUNDS: 4.08% missing
 - B_MULTIPLE_CAE: 1.23% missing
 - LOTS_NUMBER: 0.67% missing
 - TOP_TYPE: 0.17% missing
 - MAIN_ACTIVITY: 0.00% missing

📊 STEP 2: CATEGORICAL CARDINALITY PROFILING
Unique values per categorical feature:
 - B_MULTIPLE_CAE: 3 unique values
 - B_EU_FUNDS: 3 unique values
 - TOP_TYPE: 10 unique values
 - ISO_COUNTRY_CODE: 33 unique values
 - B_FRA_AGREEMENT: 2 unique values
 - B_GPA: 3 unique values
 - TYPE_OF_CONTRACT: 3 unique values
 - CAE_TYPE: 10 unique values
 - CRIT_CODE: 3 unique values
 - B_ACCELERATED: 2 unique values
 - MAIN_ACTIVITY: 65 unique values
 - CRIT_PRICE_WEIGHT: 2041 unique values

📊 STEP 3: CORRELATION & TARGET LEAKAGE CHECK
Correlation with TARGET_NO

In [4]:
df_ultimate.head()

,B_MULTIPLE_CAE,B_EU_FUNDS,TOP_TYPE,ISO_COUNTRY_CODE,B_FRA_AGREEMENT,B_GPA,YEAR,TYPE_OF_CONTRACT,CAE_TYPE,CRIT_CODE,B_ACCELERATED,MAIN_ACTIVITY,CRIT_PRICE_WEIGHT,LOTS_NUMBER,TARGET_NOT_AWARDED
0,N,N,OPE,SE,Y,Y,2021,U,3,L,NaN,General public\services,NaN,4.0,0
1,N,N,OPE,CH,N,Y,2023,S,3,M,NaN,"Recreation, culture and religion",NaN,1.0,0
2,N,N,OPE,ES,N,N,2022,S,3,L,NaN,General public\services,NaN,1.0,0
3,N,N,OPE,FR,Y,Y,2023,U,6,M,NaN,Health,NaN,243.0,0
4,N,N,OPE,RO,N,N,2020,U,1,L,NaN,Defence,NaN,96.0,1
